# Azure AI Content Understanding — batch OCR to markdown

Sends **every PDF in this folder, one by one,** to an Azure **Content Understanding** analyzer (`prebuilt-layout` = OCR + layout) and writes each document's extracted text as a **markdown file** (`results/<name>.md`, plus the full result as `results/<name>_result.json`). The markdown is then printed / rendered right here.

1. Install · 2. Configure · 3. Create client · 4. List analyzers · 5. Analyze all PDFs · 6. Print the markdown · 7. (optional) Vision-LLM pass for hierarchy charts.

The API key is read from the `AZURE_CU_KEY` environment variable (or prompted) — it is never stored in this notebook.

## 1. Install dependencies

In [ ]:
"""Install the Content Understanding SDK (pre-release), Azure identity and pandas into the kernel."""
%pip install --pre --quiet azure-ai-contentunderstanding
%pip install --quiet azure-identity pandas

## 2. Configuration

The API key is read from the environment variable `AZURE_CU_KEY`; if it isn't set you are prompted for it. **Never paste a key into this notebook.**

In [14]:
"""Configuration: endpoint, API version, key, analyzer and folders.

    - ENDPOINT / API_VERSION: the Azure AI Foundry resource to call.
    - KEY: read from the AZURE_CU_KEY env var, else prompted via getpass (never stored in the notebook).
    - ANALYZER_ID: 'prebuilt-layout' = OCR + layout (words, lines, tables, checkboxes), no custom fields.
    - PDF_DIR / OUT_DIR / PDFS: folder to scan, where results are written, and the sorted list of PDFs found."""
import os, getpass
from pathlib import Path

# --- Resource -------------------------------------------------------------
ENDPOINT = "https://<your-resource>.services.ai.azure.com/"
API_VERSION = "2025-11-01"

# Key comes from the AZURE_CU_KEY env var, else you're prompted (never stored in the notebook).
KEY = os.environ.get("AZURE_CU_KEY", "").strip() or getpass.getpass("Azure CU key: ").strip()

# --- What to analyze ----------------------------------------------------
# 'prebuilt-layout'   -> OCR + layout + words/lines/tables/figures (no custom fields)
# 'prebuilt-document' -> same, plus document-level extras; use your own analyzer id for custom fields.
# NOTE: 'prebuilt-documentAnalyzer' (used in the original notebook) does not exist on this resource.
ANALYZER_ID = "prebuilt-layout"

# Folder with the PDFs (this notebook's folder by default) and where results are written.
PDF_DIR = Path.cwd()
if not list(PDF_DIR.glob("*.pdf")):
    PDF_DIR = Path(r"C:\Users\<you>\Documents\LearnCode\azure_content_understanding")
OUT_DIR = PDF_DIR / "results"

assert ENDPOINT.startswith("http"), "Set a valid ENDPOINT"
assert KEY, "No API key found — set the AZURE_CU_KEY env var"
PDFS = sorted(PDF_DIR.glob("*.pdf"))
assert PDFS, f"No PDFs in {PDF_DIR}"
print(f"Analyzer : {ANALYZER_ID}")
print(f"Folder   : {PDF_DIR}  ({len(PDFS)} PDFs)")
for p in PDFS:
    print(f"  - {p.name}  ({p.stat().st_size:,} bytes)")

Analyzer : prebuilt-layout
Folder   : C:\Users\<you>\Documents\LearnCode\azure_content_understanding  (14 PDFs)
  - 01_polish_names_image_v1_register.pdf  (311,874 bytes)
  - 01_polish_names_image_v2_account_form.pdf  (289,349 bytes)
  - 02_watermark_v1_confidential_factsheet.pdf  (73,945 bytes)
  - 02_watermark_v2_draft_tiled_letter.pdf  (74,857 bytes)
  - 03_signature_overlap_v1_signatories.pdf  (165,070 bytes)
  - 03_signature_overlap_v2_subscription_block.pdf  (128,732 bytes)
  - 04_table_v1_trade_blotter_text.pdf  (73,924 bytes)
  - 04_table_v2_trade_order_image.pdf  (294,964 bytes)
  - 05_fund_hierarchy_v1_master_feeder.pdf  (75,087 bytes)
  - 05_fund_hierarchy_v2_umbrella_subfunds.pdf  (75,520 bytes)
  - 06_form_checkbox_v1_acroform_kyc.pdf  (80,738 bytes)
  - 06_form_checkbox_v2_flat_tickbox_form.pdf  (73,654 bytes)
  - 07_handwritten_order_v1_please_process.pdf  (122,313 bytes)
  - 07_handwritten_order_v2_certify_filled_by_hand.pdf  (159,741 bytes)


## 3. Create the client

In [15]:
"""Create the ContentUnderstandingClient from ENDPOINT, KEY and API_VERSION. No request is sent here, so a bad key only fails later."""
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential

credential = AzureKeyCredential(KEY)
# Alternative (no key): from azure.identity import DefaultAzureCredential; credential = DefaultAzureCredential()

client = ContentUnderstandingClient(
    endpoint=ENDPOINT,
    credential=credential,
    api_version=API_VERSION,
)
print("client ready")

client ready


## 4. List analyzers on this resource

Confirms the analyzer IDs that exist before calling analyze — a wrong `ANALYZER_ID` returns 404, a wrong key or endpoint returns 401. Uses `GET {endpoint}/contentunderstanding/analyzers?api-version=...`.

In [16]:
"""List the analyzers that exist on this resource (REST GET /contentunderstanding/analyzers).

    A wrong ANALYZER_ID returns 404, and a wrong key or endpoint returns 401, so this is the first real
    check that the credentials work. Warns if ANALYZER_ID is not in the list."""
import json, urllib.request

_url = f"{ENDPOINT.rstrip('/')}/contentunderstanding/analyzers?api-version={API_VERSION}"
_req = urllib.request.Request(_url, headers={"Ocp-Apim-Subscription-Key": KEY})
with urllib.request.urlopen(_req) as _resp:
    _data = json.load(_resp)

_analyzers = _data.get("value", _data if isinstance(_data, list) else [])
print(f"{len(_analyzers)} analyzer(s):\n")
for a in _analyzers:
    print(f"  {str(a.get('analyzerId')):40}  status={a.get('status')}  {a.get('description', '')}")

_ids = [a.get("analyzerId") for a in _analyzers]
if _ids and ANALYZER_ID not in _ids:
    print(
        f"\n[!] ANALYZER_ID={ANALYZER_ID!r} is not listed. "
        "If the analyze call 404s, set it to one of the above and re-run from cell 1."
    )

88 analyzer(s):

  prebuilt-tax.us.w4                        status=ready  Extract tax US w4 document fields.
  prebuilt-tax.us.1099NEC                   status=ready  Extract tax US 1099 nec document fields.
  prebuilt-purchaseOrder                    status=ready  Extract purchase order document fields.
  prebuilt-creditCard                       status=ready  Extract credit card document fields.
  prebuilt-tax.us.1099QA                    status=ready  Extract tax US 1099 qa document fields.
  prebuilt-audio                            status=ready  Transcribe conversations.
  prebuilt-tax.us.1099H                     status=ready  Extract tax US 1099 h document fields.
  prebuilt-tax.us.1099S                     status=ready  Extract tax US 1099 s document fields.
  prebuilt-tax.us.1099LS                    status=ready  Extract tax US 1099 ls document fields.
  prebuilt-tax.us.1099DA                    status=ready  Extract tax US 1099 da document fields.
  prebuilt-tax.us.1099G   

## 5. Analyze every PDF, one by one

For each file: upload the bytes, wait for the analyzer, then save `results/<name>_result.json` (full result) and `results/<name>.md` (the markdown). A failure on one file is recorded and the loop moves on.

In [17]:
"""Analyze every PDF one by one (sequentially).

    For each file: upload the bytes, wait for the poller, then save results/<name>_result.json (full result)
    and results/<name>.md (markdown). A failure on one file is recorded and the loop moves on.
    Fills `results` (file name -> result dict) and shows `summary_df` (status, seconds, pages, words, lines,
    tables, markdown size per file)."""
import json, time
import pandas as pd

OUT_DIR.mkdir(exist_ok=True)
results = {}   # file name -> full result dict
rows = []      # one summary row per file

for i, pdf in enumerate(PDFS, 1):
    print(f"[{i}/{len(PDFS)}] {pdf.name} ... ", end="", flush=True)
    t0 = time.time()
    try:
        poller = client.begin_analyze_binary(analyzer_id=ANALYZER_ID, binary_input=pdf.read_bytes())
        r = poller.result().as_dict()
    except Exception as e:
        print(f"FAILED ({type(e).__name__}: {str(e).splitlines()[0][:120]})")
        rows.append({"file": pdf.name, "status": "failed", "secs": round(time.time() - t0, 1), "error": str(e).splitlines()[0][:120]})
        continue

    secs = time.time() - t0
    results[pdf.name] = r
    (OUT_DIR / f"{pdf.stem}_result.json").write_text(json.dumps(r, indent=2), encoding="utf-8")
    markdown = "\n\n".join(c.get("markdown") or "" for c in r.get("contents", []))
    (OUT_DIR / f"{pdf.stem}.md").write_text(markdown, encoding="utf-8")

    cs = r.get("contents", []) or []
    pgs = [p for c in cs for p in (c.get("pages") or [])]
    rows.append({
        "file": pdf.name, "status": "ok", "secs": round(secs, 1),
        "pages": len(pgs),
        "words": sum(len(p.get("words") or []) for p in pgs),
        "lines": sum(len(p.get("lines") or []) for p in pgs),
        "tables": sum(len(c.get("tables") or []) for c in cs),
        "md_chars": len(markdown),
    })
    print(f"ok in {secs:.1f}s ({len(markdown):,} chars of markdown)")

summary_df = pd.DataFrame(rows)
print(f"\nresults written to {OUT_DIR.resolve()}")
summary_df

[1/14] 01_polish_names_image_v1_register.pdf ... ok in 3.7s (914 chars of markdown)
[2/14] 01_polish_names_image_v2_account_form.pdf ... ok in 3.5s (595 chars of markdown)
[3/14] 02_watermark_v1_confidential_factsheet.pdf ... ok in 3.4s (2,516 chars of markdown)
[4/14] 02_watermark_v2_draft_tiled_letter.pdf ... ok in 3.3s (1,173 chars of markdown)
[5/14] 03_signature_overlap_v1_signatories.pdf ... ok in 3.3s (274 chars of markdown)
[6/14] 03_signature_overlap_v2_subscription_block.pdf ... ok in 3.4s (338 chars of markdown)
[7/14] 04_table_v1_trade_blotter_text.pdf ... ok in 3.4s (961 chars of markdown)
[8/14] 04_table_v2_trade_order_image.pdf ... ok in 3.4s (567 chars of markdown)
[9/14] 05_fund_hierarchy_v1_master_feeder.pdf ... ok in 3.4s (505 chars of markdown)
[10/14] 05_fund_hierarchy_v2_umbrella_subfunds.pdf ... ok in 3.5s (570 chars of markdown)
[11/14] 06_form_checkbox_v1_acroform_kyc.pdf ... ok in 3.3s (266 chars of markdown)
[12/14] 06_form_checkbox_v2_flat_tickbox_form.pdf .

,file,status,secs,pages,words,lines,tables,md_chars
0,01_polish_names_image_v1_register.pdf,ok,3.7,1,109,22,1,914
1,01_polish_names_image_v2_account_form.pdf,ok,3.5,1,60,17,1,595
2,02_watermark_v1_confidential_factsheet.pdf,ok,3.4,1,572,32,0,2516
3,02_watermark_v2_draft_tiled_letter.pdf,ok,3.3,2,205,32,0,1173
4,03_signature_overlap_v1_signatories.pdf,ok,3.3,1,44,19,0,274
5,03_signature_overlap_v2_subscription_block.pdf,ok,3.4,1,66,11,0,338
6,04_table_v1_trade_blotter_text.pdf,ok,3.4,1,82,52,1,961
7,04_table_v2_trade_order_image.pdf,ok,3.4,1,57,29,1,567
8,05_fund_hierarchy_v1_master_feeder.pdf,ok,3.4,1,86,20,0,505
9,05_fund_hierarchy_v2_umbrella_subfunds.pdf,ok,3.5,1,119,32,0,570


## 6. Print the markdown

`show_markdown()` renders every document's markdown (headings, tables, ☐/☒ checkboxes) one after another; `show_markdown("<file>.pdf")` renders one; add `raw=True` to see the unrendered markdown source. The same text is saved in `results/<name>.md`.

In [18]:
"""Markdown viewer helpers.

    get_markdown(name)             -> the document's markdown as a string.
    show_markdown(name=None, raw=False) -> render one document (or all when name is omitted);
                                       raw=True shows the unrendered markdown source instead."""
from IPython.display import display, Markdown

def get_markdown(name):
    """Markdown for one document (all of its `contents` joined)."""
    return "\n\n".join(c.get("markdown") or "" for c in results[name].get("contents", []))

def show_markdown(name=None, raw=False):
    """Render markdown for `name` (a key of `results`), or for every document if omitted."""
    for n in ([name] if name else list(results)):
        display(Markdown(f"---\n### {n}"))
        text = get_markdown(n) or "_<no text extracted>_"
        display(Markdown(f"```markdown\n{text}\n```") if raw else Markdown(text))

print("Documents:", *[f"  {k}" for k in results], sep="\n")

Documents:
  01_polish_names_image_v1_register.pdf
  01_polish_names_image_v2_account_form.pdf
  02_watermark_v1_confidential_factsheet.pdf
  02_watermark_v2_draft_tiled_letter.pdf
  03_signature_overlap_v1_signatories.pdf
  03_signature_overlap_v2_subscription_block.pdf
  04_table_v1_trade_blotter_text.pdf
  04_table_v2_trade_order_image.pdf
  05_fund_hierarchy_v1_master_feeder.pdf
  05_fund_hierarchy_v2_umbrella_subfunds.pdf
  06_form_checkbox_v1_acroform_kyc.pdf
  06_form_checkbox_v2_flat_tickbox_form.pdf
  07_handwritten_order_v1_please_process.pdf
  07_handwritten_order_v2_certify_filled_by_hand.pdf


In [19]:
"""Render the markdown of a single document. Change the file name to view another one."""
# One document (change the name), rendered:
show_markdown("01_polish_names_image_v2_account_form.pdf")

---
### 01_polish_names_image_v2_account_form.pdf

# WNIOSEK O OTWARCIE RACHUNKU Account Opening Application / Demande d'ouverture de compte


<table>
<tr>
<td>Imię i nazwisko:</td>
<td>Agnieszka Wiśniewska-Ślązak</td>
</tr>
<tr>
<td>Nazwisko rodowe:</td>
<td>Błaszczyk</td>
</tr>
<tr>
<td>Miejsce urodzenia:</td>
<td>Łódź</td>
</tr>
<tr>
<td>Adres:</td>
<td>ul. Świętojańska 12/4, 81-368 Gdynia</td>
</tr>
<tr>
<td>Reprezentant:</td>
<td>Przemysław Gołębiowski</td>
</tr>
<tr>
<td>Beneficiary / Bénéficiaire:</td>
<td>Hélène Dubois-Côté</td>
</tr>
<tr>
<td>Contact:</td>
<td>Björn Åkesson, Göteborg</td>
</tr>
</table>


Data / Date: 12.06.2026


In [20]:
"""Render the markdown of every document, one after another."""
# Every document, one after another:
show_markdown()

---
### 01_polish_names_image_v1_register.pdf

# Rejestr Uczestników / Investor Register


## Register of Beneficial Owners - Fund Units


<table>
<caption>As of 30 June 2026</caption>
<tr>
<th>Investor name</th>
<th>Account / IBAN</th>
</tr>
<tr>
<td>Zażółć Gęślą-Jaźń</td>
<td>PL61 1090 1014 0000 0712 1981 2874</td>
</tr>
<tr>
<td>Łukasz Wójcik</td>
<td>PL27 1140 2004 0000 3002 0135 5387</td>
</tr>
<tr>
<td>Małgorzata Szymańska</td>
<td>PL10 1050 1214 1000 0090 3200 0500</td>
</tr>
<tr>
<td>Józef Kowalczyk-Żelazny</td>
<td>PL83 1020 1026 0000 0422 7020 1111</td>
</tr>
<tr>
<td>Ćwikła Dąbrowska-Nożyk</td>
<td>PL66 1240 6292 1111 0010 5090 5432</td>
</tr>
<tr>
<td>José Ñúñez Peña</td>
<td>ES91 2100 0418 4502 0005 1332</td>
</tr>
<tr>
<td>François Lefèvre-Müller</td>
<td>FR14 2004 1010 0505 0001 3M02 606</td>
</tr>
<tr>
<td>Søren Ærøskøbing</td>
<td>DK50 0040 0440 1162 43</td>
</tr>
</table>


Prepared by: Wojciech Przybylski, Dział Obsługi Klienta


---
### 01_polish_names_image_v2_account_form.pdf

# WNIOSEK O OTWARCIE RACHUNKU Account Opening Application / Demande d'ouverture de compte


<table>
<tr>
<td>Imię i nazwisko:</td>
<td>Agnieszka Wiśniewska-Ślązak</td>
</tr>
<tr>
<td>Nazwisko rodowe:</td>
<td>Błaszczyk</td>
</tr>
<tr>
<td>Miejsce urodzenia:</td>
<td>Łódź</td>
</tr>
<tr>
<td>Adres:</td>
<td>ul. Świętojańska 12/4, 81-368 Gdynia</td>
</tr>
<tr>
<td>Reprezentant:</td>
<td>Przemysław Gołębiowski</td>
</tr>
<tr>
<td>Beneficiary / Bénéficiaire:</td>
<td>Hélène Dubois-Côté</td>
</tr>
<tr>
<td>Contact:</td>
<td>Björn Åkesson, Göteborg</td>
</tr>
</table>


Data / Date: 12.06.2026


---
### 02_watermark_v1_confidential_factsheet.pdf

...

# Redemption Request Form

☐
Full redemption

☒
Partial redemption

☐
Switch to another sub-fund

☒
Pay proceeds by wire transfer

☐
Pay proceeds by cheque

☒
Terms and conditions accepted

Units to redeem: 2,450.000
Fund: Northbridge Global Equity Fund


---
### 07_handwritten_order_v1_please_process.pdf

# TRADE ORDER INSTRUCTION Halden & Roe Securities - Dealing Desk | dealing@haldenroe.example

Account No.:

Side (Buy/Sell):

Security / ISIN:

Quantity:

Order type:

Limit price:

Special instructions / certification

Please process Order

Client name (print): Tomasz Wróbel

T. Wrobel

Authorised signature

Date: 24/09/2026


---
### 07_handwritten_order_v2_certify_filled_by_hand.pdf

# TRADE ORDER INSTRUCTION Halden & Roe Securities - Dealing Desk | dealing@haldenroe.example


<table>
<tr>
<td>Account No.:</td>
<td>ACC-77120</td>
</tr>
<tr>
<td>Side (Buy/Sell):</td>
<td>BUY</td>
</tr>
<tr>
<td>Security / ISIN:</td>
<td>MSFT US5949181045</td>
</tr>
<tr>
<td>Quantity:</td>
<td>1,200</td>
</tr>
<tr>
<td>Order type:</td>
<td>Limit</td>
</tr>
<tr>
<td>Limit price:</td>
<td>421.50</td>
</tr>
</table>


Special instructions / certification

I certify the order

Client name (print): Tomasz Wróbel

Tomasz W.

Authorised signature

Date: 24/09/2026


## 7. (Optional) Vision-LLM processing for hierarchy documents (Azure OpenAI GPT)

`prebuilt-layout` reads the **text** of an org / fund-structure chart but loses the **arrows**: the whole diagram comes back as one figure with a flat caption. To recover *who relates to whom* we send the rendered page image to a **vision LLM** (Azure OpenAI `gpt-6-astra`, Responses API) and ask for a graph — **nodes** (entities) and **edges** (relationships) — as schema-validated JSON.

Pipeline per hierarchy PDF:
1. Render each page to a PNG (PyMuPDF).
2. Send the PNG **plus the OCR markdown from section 5** (so names are spelled exactly) to the model, constrained to a JSON schema (`text.format` = strict `json_schema`).
3. **Validate** against the OCR text: every node name should appear in it, every edge must reference known nodes, and unconnected nodes are flagged.
4. Show `nodes_df` / `edges_df`, an indented tree, and a Mermaid diagram; save `results/<name>_hierarchy.json`.

Needs the Azure OpenAI key (`AZURE_OPENAI_KEY` env var, otherwise you are prompted). Sections 1-3 must have been run first (this reuses `results` and `PDFS`).

In [ ]:
"""Install the OpenAI SDK (used for Azure OpenAI) and PyMuPDF (used to render PDF pages to images)."""
%pip install --quiet openai pymupdf

In [21]:
"""Vision-LLM configuration: Azure OpenAI client, deployment, which PDFs count as hierarchy documents, render settings."""
import os, getpass, base64, json, re
from openai import AzureOpenAI
import pymupdf

# Same Azure OpenAI resource as azure_gpt5_setup.ipynb and responses_openai.ipynb; deployment per your request.
OPENAI_ENDPOINT = "https://<your-resource>.cognitiveservices.azure.com/"
OPENAI_API_VERSION = "2025-04-01-preview"
VISION_MODEL = "gpt-6-astra"           # Azure deployment name (must support image input)

# Key comes from the AZURE_OPENAI_KEY env var, else you're prompted (never stored in the notebook).
OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY", "").strip() or getpass.getpass("Azure OpenAI key: ").strip()
vision_client = AzureOpenAI(azure_endpoint=OPENAI_ENDPOINT, api_version=OPENAI_API_VERSION, api_key=OPENAI_KEY)

RENDER_DPI = 150                   # higher = sharper small text, more tokens
MAX_PAGES = 4                      # pages sent per document

# Which documents to treat as hierarchy charts: file names starting with these prefixes.
HIERARCHY_PREFIXES = ("05_",)
HIERARCHY_DOCS = [p for p in PDFS if p.name.startswith(HIERARCHY_PREFIXES)]
print("Model :", VISION_MODEL)
print("Docs  :", *[p.name for p in HIERARCHY_DOCS], sep="\n  ")

Model : gpt-6-astra
Docs  :
  05_fund_hierarchy_v1_master_feeder.pdf
  05_fund_hierarchy_v2_umbrella_subfunds.pdf


In [22]:
"""Define the extraction step: render pages, build the prompt + JSON schema, call the model, return the parsed graph.

    render_pages(pdf_path)                       -> list of base64 PNG strings
    extract_hierarchy(pdf_path, ocr_text, ...)   -> dict {"nodes": [...], "edges": [...], "notes": str}
    """
HIERARCHY_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "Title of the chart, or empty string."},
        "nodes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "Short unique slug, e.g. 'master_fund'."},
                    "name": {"type": "string", "description": "Entity name exactly as printed in the box."},
                    "entity_type": {"type": "string", "description": "e.g. general_partner, investment_manager, master_fund, feeder_fund, umbrella_fund, sub_fund, share_class, management_company, depositary, custodian, administrator, prime_broker, investor_group, other."},
                    "jurisdiction": {"type": "string", "description": "Jurisdiction if printed (e.g. Cayman Islands), else empty string."},
                    "sublabel": {"type": "string", "description": "Any secondary text in the box, else empty string."},
                },
                "required": ["id", "name", "entity_type", "jurisdiction", "sublabel"],
                "additionalProperties": False,
            },
        },
        "edges": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "source": {"type": "string", "description": "id of the node the arrow/line starts at (the parent when there is no arrowhead)."},
                    "target": {"type": "string", "description": "id of the node the arrow/line points to."},
                    "relation": {"type": "string", "description": "Label printed on the line if any (e.g. 'manages', 'invests'); otherwise a short verb phrase implied by the chart."},
                    "label_printed": {"type": "boolean", "description": "True if the relation text is printed on the chart, False if you inferred it."},
                    "style": {"type": "string", "description": "'solid', 'dashed' or 'none/unclear'."},
                },
                "required": ["source", "target", "relation", "label_printed", "style"],
                "additionalProperties": False,
            },
        },
        "notes": {"type": "string", "description": "Anything ambiguous, unreadable, or inferred. Empty string if nothing."},
    },
    "required": ["title", "nodes", "edges", "notes"],
    "additionalProperties": False,
}

PROMPT = """You are reading a fund / corporate structure chart. Extract it as a graph.

- nodes: one per box (or standalone labelled entity). Copy names exactly as printed; do not invent entities.
- edges: one per connector line. Direction: follow the arrowhead (source -> target). If a line has no arrowhead,
  go from the entity higher on the page to the one lower (parent -> child) and say so in `notes`.
- relation: use the text printed on or next to the line when there is one (label_printed = true);
  otherwise give a short phrase implied by the chart (label_printed = false).
- Elbow connectors that split to several boxes are one edge per destination box.
- Text outside boxes (e.g. a list of service providers with no connector) is NOT an edge; make a node for it only if it is a labelled box.
- Put anything uncertain in `notes` rather than guessing.

OCR text of the same document (from a layout analyzer) - use it to spell names exactly:
-----
{ocr}
-----"""


def render_pages(pdf_path, dpi=RENDER_DPI, max_pages=MAX_PAGES):
    """Render up to `max_pages` pages of a PDF to base64-encoded PNG strings."""
    out = []
    with pymupdf.open(pdf_path) as doc:
        for page in list(doc)[:max_pages]:
            png = page.get_pixmap(dpi=dpi).tobytes("png")
            out.append(base64.standard_b64encode(png).decode("ascii"))
    return out


def extract_hierarchy(pdf_path, ocr_text, model=VISION_MODEL):
    """Send the rendered pages + OCR text to the vision model and return the parsed graph dict."""
    content = [{"type": "input_image", "image_url": f"data:image/png;base64,{b64}"} for b64 in render_pages(pdf_path)]
    content.append({"type": "input_text", "text": PROMPT.format(ocr=ocr_text or "(none)")})
    resp = vision_client.responses.create(
        model=model,
        input=[{"role": "user", "content": content}],
        text={"format": {"type": "json_schema", "name": "fund_hierarchy", "schema": HIERARCHY_SCHEMA, "strict": True}},
        reasoning={"effort": "medium"},
        max_output_tokens=16000,
    )
    if resp.status != "completed":
        raise RuntimeError(f"response {resp.status}: {getattr(resp, 'incomplete_details', None) or resp.error}")
    graph = json.loads(resp.output_text)
    graph["_usage"] = {"input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}
    return graph

In [23]:
"""Define validate_hierarchy() and the display helpers.

    validate_hierarchy(graph, ocr_text) -> list of warning strings (empty = clean). Checks that every node name
        appears in the OCR text, edges reference known nodes, ids are unique, and which nodes have no edges.
    to_mermaid(graph) -> Mermaid `graph TD` source;  print_tree(graph) -> indented parent -> child listing.
    """
import pandas as pd
from IPython.display import display, Markdown


def _norm(s):
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()


def validate_hierarchy(graph, ocr_text):
    """Cross-check the vision output against the OCR text and its own internal consistency."""
    warnings, ocr = [], _norm(ocr_text)
    ids = [n["id"] for n in graph["nodes"]]
    for dup in {i for i in ids if ids.count(i) > 1}:
        warnings.append(f"duplicate node id: {dup}")
    for n in graph["nodes"]:
        if _norm(n["name"]) not in ocr:
            warnings.append(f"node name not found in OCR text: {n['name']!r}")
    known = set(ids)
    for e in graph["edges"]:
        for end in ("source", "target"):
            if e[end] not in known:
                warnings.append(f"edge references unknown node {e[end]!r}: {e['source']} -> {e['target']}")
    linked = {e["source"] for e in graph["edges"]} | {e["target"] for e in graph["edges"]}
    for n in graph["nodes"]:
        if n["id"] not in linked:
            warnings.append(f"node has no edges: {n['name']!r}")
    return warnings


def to_mermaid(graph):
    """Mermaid flowchart source for the graph."""
    lines = ["graph TD"]
    for n in graph["nodes"]:
        label = n["name"].replace('"', "'") + (f"<br/>{n['sublabel']}".replace('"', "'") if n["sublabel"] else "")
        lines.append(f'    {n["id"]}["{label}"]')
    for e in graph["edges"]:
        arrow = "-.->" if e["style"] == "dashed" else "-->"
        lines.append(f'    {e["source"]} {arrow}|{e["relation"].replace("|", "/")}| {e["target"]}')
    return "\n".join(lines)


def print_tree(graph):
    """Indented listing: each source with the targets it points to, and the relation."""
    name = {n["id"]: n["name"] for n in graph["nodes"]}
    children = {}
    for e in graph["edges"]:
        children.setdefault(e["source"], []).append(e)
    targets = {e["target"] for e in graph["edges"]}
    roots = [n["id"] for n in graph["nodes"] if n["id"] not in targets] or [n["id"] for n in graph["nodes"]]

    def walk(i, depth, seen):
        for e in children.get(i, []):
            print("    " * depth + f"-[{e['relation']}]-> {name.get(e['target'], e['target'])}")
            if e["target"] not in seen:
                walk(e["target"], depth + 1, seen | {e["target"]})
    for r in roots:
        print(name.get(r, r)); walk(r, 1, {r})

In [24]:
"""Run the vision LLM over every hierarchy document, one by one.

    For each file: extract -> validate -> show nodes/edges tables, tree and Mermaid source -> save
    results/<name>_hierarchy.json. Fills `hierarchies` (file name -> graph dict).
    """
hierarchies = {}
for pdf in HIERARCHY_DOCS:
    ocr_text = get_markdown(pdf.name) if pdf.name in results else ""
    print("=" * 100); print(pdf.name); print("=" * 100)
    try:
        graph = extract_hierarchy(pdf, ocr_text)
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}\n"); continue
    hierarchies[pdf.name] = graph
    (OUT_DIR / f"{pdf.stem}_hierarchy.json").write_text(json.dumps(graph, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"{len(graph['nodes'])} nodes, {len(graph['edges'])} edges | tokens in/out: "
          f"{graph['_usage']['input_tokens']:,}/{graph['_usage']['output_tokens']:,}")
    warns = validate_hierarchy(graph, ocr_text)
    print("validation:", "clean" if not warns else "")
    for w in warns:
        print("  -", w)
    if graph["notes"]:
        print("model notes:", graph["notes"])
    display(pd.DataFrame(graph["nodes"]))
    display(pd.DataFrame(graph["edges"]))
    print_tree(graph)
    print("\nMermaid:\n" + to_mermaid(graph) + "\n")

05_fund_hierarchy_v1_master_feeder.pdf
11 nodes, 7 edges | tokens in/out: 3,279/980
validation: 
  - node has no edges: 'Prime Broker: Goldman Sachs'
  - node has no edges: 'Administrator: Apex Fund Services'
  - node has no edges: 'Custodian: BNP Paribas'
model notes: The three feeder connections share an upward arrow to the Master Fund and a single printed 'invests' label. Service-provider boxes have no connectors, so no edges are included for them.


,id,name,entity_type,jurisdiction,sublabel
0,capital_gp,Orion Capital GP Ltd,general_partner,,General Partner / Investment Manager
1,master_fund,Orion Master Fund LP,master_fund,Cayman Islands,Cayman Islands
2,onshore_feeder,Orion Onshore Feeder LP,feeder_fund,Delaware,Delaware - US taxable
3,offshore_feeder,Orion Offshore Feeder Ltd,feeder_fund,Cayman,Cayman - non-US / tax-exempt
4,europe_feeder,Orion Europe Feeder SICAV,feeder_fund,Luxembourg,Luxembourg - UCITS-like
5,us_investors,US Taxable Investors,investor_group,,
6,non_us_investors,Non-US Investors,investor_group,,
7,eu_investors,EU Investors,investor_group,,
8,prime_broker,Prime Broker: Goldman Sachs,prime_broker,,
9,administrator,Administrator: Apex Fund Services,administrator,,


,source,target,relation,label_printed,style
0,capital_gp,master_fund,manages,True,solid
1,onshore_feeder,master_fund,invests,True,solid
2,offshore_feeder,master_fund,invests,True,solid
3,europe_feeder,master_fund,invests,True,solid
4,us_investors,onshore_feeder,invests in,False,solid
5,non_us_investors,offshore_feeder,invests in,False,solid
6,eu_investors,europe_feeder,invests in,False,solid


Orion Capital GP Ltd
    -[manages]-> Orion Master Fund LP
US Taxable Investors
    -[invests in]-> Orion Onshore Feeder LP
        -[invests]-> Orion Master Fund LP
Non-US Investors
    -[invests in]-> Orion Offshore Feeder Ltd
        -[invests]-> Orion Master Fund LP
EU Investors
    -[invests in]-> Orion Europe Feeder SICAV
        -[invests]-> Orion Master Fund LP
Prime Broker: Goldman Sachs
Administrator: Apex Fund Services
Custodian: BNP Paribas

Mermaid:
graph TD
    capital_gp["Orion Capital GP Ltd<br/>General Partner / Investment Manager"]
    master_fund["Orion Master Fund LP<br/>Cayman Islands"]
    onshore_feeder["Orion Onshore Feeder LP<br/>Delaware - US taxable"]
    offshore_feeder["Orion Offshore Feeder Ltd<br/>Cayman - non-US / tax-exempt"]
    europe_feeder["Orion Europe Feeder SICAV<br/>Luxembourg - UCITS-like"]
    us_investors["US Taxable Investors"]
    non_us_investors["Non-US Investors"]
    eu_investors["EU Investors"]
    prime_broker["Prime Broker: Goldman S

,id,name,entity_type,jurisdiction,sublabel
0,management_company,Aldgate ManCo (Ireland) Ltd,management_company,,Management Company
1,umbrella,Aldgate UCITS ICAV,umbrella_fund,Ireland,Umbrella - Ireland
2,depositary,State Street Custodial Services,depositary,,Depositary
3,global_equity,Global Equity Fund,sub_fund,,Sub-fund
4,euro_bond,Euro Bond Fund,sub_fund,,Sub-fund
5,multi_asset_income,Multi-Asset Income Fund,sub_fund,,Sub-fund
6,global_equity_a,Class A,share_class,,(EUR Acc)
7,global_equity_i,Class I,share_class,,(USD Dist)
8,global_equity_z,Class Z,share_class,,(GBP Acc)
9,euro_bond_a,Class A,share_class,,(EUR Acc)


,source,target,relation,label_printed,style
0,management_company,umbrella,manages,False,solid
1,umbrella,depositary,has depositary,False,solid
2,umbrella,global_equity,has sub-fund,False,solid
3,umbrella,euro_bond,has sub-fund,False,solid
4,umbrella,multi_asset_income,has sub-fund,False,solid
5,global_equity,global_equity_a,has share class,False,solid
6,global_equity,global_equity_i,has share class,False,solid
7,global_equity,global_equity_z,has share class,False,solid
8,euro_bond,euro_bond_a,has share class,False,solid
9,euro_bond,euro_bond_i,has share class,False,solid


Aldgate ManCo (Ireland) Ltd
    -[manages]-> Aldgate UCITS ICAV
        -[has depositary]-> State Street Custodial Services
        -[has sub-fund]-> Global Equity Fund
            -[has share class]-> Class A
            -[has share class]-> Class I
            -[has share class]-> Class Z
        -[has sub-fund]-> Euro Bond Fund
            -[has share class]-> Class A
            -[has share class]-> Class I
            -[has share class]-> Class Z
        -[has sub-fund]-> Multi-Asset Income Fund
            -[has share class]-> Class A
            -[has share class]-> Class I
            -[has share class]-> Class Z

Mermaid:
graph TD
    management_company["Aldgate ManCo (Ireland) Ltd<br/>Management Company"]
    umbrella["Aldgate UCITS ICAV<br/>Umbrella - Ireland"]
    depositary["State Street Custodial Services<br/>Depositary"]
    global_equity["Global Equity Fund<br/>Sub-fund"]
    euro_bond["Euro Bond Fund<br/>Sub-fund"]
    multi_asset_income["Multi-Asset Income Fund<br/>Su

In [25]:
"""Render the Mermaid diagram of one hierarchy as a picture (JupyterLab 4+ renders ```mermaid blocks; other front-ends show the source)."""
HIER_NAME = next(iter(hierarchies), None)       # <- change to any key of `hierarchies`
if HIER_NAME:
    display(Markdown("```mermaid\n" + to_mermaid(hierarchies[HIER_NAME]) + "\n```"))
else:
    print("no hierarchy results yet - run the previous cell")

```mermaid
graph TD
    capital_gp["Orion Capital GP Ltd<br/>General Partner / Investment Manager"]
    master_fund["Orion Master Fund LP<br/>Cayman Islands"]
    onshore_feeder["Orion Onshore Feeder LP<br/>Delaware - US taxable"]
    offshore_feeder["Orion Offshore Feeder Ltd<br/>Cayman - non-US / tax-exempt"]
    europe_feeder["Orion Europe Feeder SICAV<br/>Luxembourg - UCITS-like"]
    us_investors["US Taxable Investors"]
    non_us_investors["Non-US Investors"]
    eu_investors["EU Investors"]
    prime_broker["Prime Broker: Goldman Sachs"]
    administrator["Administrator: Apex Fund Services"]
    custodian["Custodian: BNP Paribas"]
    capital_gp -->|manages| master_fund
    onshore_feeder -->|invests| master_fund
    offshore_feeder -->|invests| master_fund
    europe_feeder -->|invests| master_fund
    us_investors -->|invests in| onshore_feeder
    non_us_investors -->|invests in| offshore_feeder
    eu_investors -->|invests in| europe_feeder
```